# UsedCarAI — 02 Random Forest


In [ ]:
from pathlib import Path
import os
import sys

# Discover CPE-306 when opened from the project or UsedcarAi directory.
# Set USEDCAR_PROJECT_DIR to use a different checkout location.
_candidates = (Path.cwd(), *Path.cwd().parents)
_default_project = next(
    (p for p in _candidates if (p / "dataset").is_dir() and (p / "app").is_dir()),
    Path("/home/regen/Works/CPE-306"),
)
PROJECT_DIR = Path(os.environ.get("USEDCAR_PROJECT_DIR", _default_project)).expanduser().resolve()
DATASET_DIR = Path(os.environ.get("DATASET_DIR", PROJECT_DIR / "dataset")).expanduser().resolve()
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")
print("Python:", sys.executable)
print("Dataset:", DATASET_DIR)
print("Use the Python kernel from the project's ML environment.")


In [ ]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, median_absolute_error

warnings.filterwarnings("ignore")

PROJECT_ROOT = DATASET_DIR
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "random_forest"
METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
PRED_DIR = PROJECT_ROOT / "results" / "predictions"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

for d in [MODEL_DIR, METRICS_DIR, PRED_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

for p in [
    PROCESSED_DIR / "train.csv",
    PROCESSED_DIR / "validation.csv",
    PROCESSED_DIR / "test.csv",
]:
    if not p.exists():
        raise FileNotFoundError(f"ไม่พบไฟล์ {p}. ให้รัน 01_data_cleaning.ipynb ให้เสร็จก่อน")

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

## Feature Set

**Categorical:** brand, model, sub_model, fuel_type, transmission, body_type, color, province, location, seller_type

**Numeric:** model_year, mileage, engine_size, number_of_seats, vehicle_age, vehicle_age_sq, mileage_log1p, mileage_per_year

ไม่ใช้:
- `price` (target)
- `price_extreme_flag` (target leakage)
- listing/source/image/description/timestamp fields


In [ ]:
TARGET = "price"

CATEGORICAL_FEATURES = [
    "brand", "model", "sub_model", "fuel_type", "transmission",
    "body_type", "color", "province", "location", "seller_type"
]
NUMERIC_FEATURES = [
    "model_year", "mileage", "engine_size", "number_of_seats",
    "vehicle_age", "vehicle_age_sq", "mileage_log1p", "mileage_per_year"
]
FEATURES = CATEGORICAL_FEATURES + NUMERIC_FEATURES

missing = [c for c in FEATURES + [TARGET] if c not in train_df.columns]
if missing:
    raise ValueError(f"ขาดคอลัมน์: {missing}")

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_val, y_val = val_df[FEATURES], val_df[TARGET]
X_test, y_test = test_df[FEATURES], test_df[TARGET]

print("Features:", len(FEATURES))

## Preprocessing + Baseline Random Forest

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=5,
        sparse_output=True
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC_FEATURES),
    ("cat", categorical_pipe, CATEGORICAL_FEATURES),
])

rf = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=2,
    max_features="sqrt"
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", rf),
])

t0 = time.time()
pipeline.fit(X_train, y_train)
baseline_seconds = time.time() - t0
print(f"Baseline training: {baseline_seconds:.1f} sec")

## Metrics

In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    mask = denom != 0
    return float(100 * np.mean(
        2 * np.abs(y_pred[mask] - y_true[mask]) / denom[mask]
    ))

def metrics(y_true, y_pred):
    return {
        "MAE_THB": float(mean_absolute_error(y_true, y_pred)),
        "RMSE_THB": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)),
        "sMAPE_percent": smape(y_true, y_pred),
        "MedianAE_THB": float(median_absolute_error(y_true, y_pred)),
    }

In [ ]:
RUN_TUNING = True
N_ITER = 6
CV = 3

if RUN_TUNING:
    search = RandomizedSearchCV(
        pipeline,
        param_distributions={
            "model__n_estimators": [150, 200, 300],
            "model__max_depth": [None, 15, 25, 35],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": ["sqrt", 0.5, 0.8],
        },
        n_iter=N_ITER,
        scoring="neg_mean_absolute_error",
        cv=CV,
        random_state=42,
        n_jobs=-1,
        verbose=2,
        refit=True
    )
    t0 = time.time()
    search.fit(X_train, y_train)
    tuning_seconds = time.time() - t0
    best_pipeline = search.best_estimator_
    print(f"Tuning time: {tuning_seconds/60:.1f} min")
    print("Best CV MAE:", -search.best_score_)
    print("Best params:", search.best_params_)
else:
    best_pipeline = pipeline
    tuning_seconds = 0.0
    print("Tuning skipped; using baseline.")

In [ ]:
val_pred = best_pipeline.predict(X_val)
test_pred = best_pipeline.predict(X_test)

val_metrics = metrics(y_val, val_pred)
test_metrics = metrics(y_test, test_pred)

print("VALIDATION")
display(pd.DataFrame([val_metrics]))
print("TEST")
display(pd.DataFrame([test_metrics]))

pd.DataFrame([
    {"model": "Random Forest", "split": "validation", **val_metrics},
    {"model": "Random Forest", "split": "test", **test_metrics},
]).to_csv(METRICS_DIR / "random_forest_metrics.csv", index=False, encoding="utf-8-sig")

## Error by Price Band

In [ ]:
def price_band_report(y_true, y_pred):
    tmp = pd.DataFrame({"actual": y_true.to_numpy(), "predicted": y_pred})
    bins = [-np.inf, 300_000, 600_000, 1_000_000, np.inf]
    labels = ["<300k", "300k-600k", "600k-1m", ">1m"]
    tmp["price_band"] = pd.cut(tmp["actual"], bins=bins, labels=labels)
    rows = []
    for band, g in tmp.groupby("price_band", observed=True):
        m = metrics(g["actual"], g["predicted"])
        rows.append({"price_band": str(band), "n": len(g), **m})
    return pd.DataFrame(rows)

band_metrics = price_band_report(y_test, test_pred)
display(band_metrics)
band_metrics.to_csv(METRICS_DIR / "random_forest_test_by_price_band.csv",
                    index=False, encoding="utf-8-sig")

## Feature Importance

In [ ]:
pre = best_pipeline.named_steps["preprocessor"]
model = best_pipeline.named_steps["model"]

fi = pd.DataFrame({
    "feature": pre.get_feature_names_out(),
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(fi.head(30))
fi.to_csv(METRICS_DIR / "random_forest_feature_importance.csv",
          index=False, encoding="utf-8-sig")

top = fi.head(20).sort_values("importance")
plt.figure(figsize=(10, 7))
plt.barh(top["feature"], top["importance"])
plt.xlabel("Importance")
plt.title("Random Forest — Top 20 Features")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "random_forest_feature_importance.png", dpi=180)
plt.show()

## Test Predictions

In [ ]:
pred_df = test_df[[
    "listing_id", "brand", "model", "sub_model",
    "model_year", "mileage", "price"
]].copy()

pred_df["predicted_price"] = test_pred
pred_df["absolute_error"] = np.abs(pred_df["price"] - pred_df["predicted_price"])
pred_df["error_percent"] = (
    pred_df["absolute_error"] / pred_df["price"].replace(0, np.nan) * 100
)

pred_df.to_csv(PRED_DIR / "random_forest_test_predictions.csv",
               index=False, encoding="utf-8-sig")

plt.figure(figsize=(8, 8))
plt.scatter(y_test, test_pred, s=8, alpha=0.35)
lo = float(min(y_test.min(), test_pred.min()))
hi = float(max(y_test.max(), test_pred.max()))
plt.plot([lo, hi], [lo, hi])
plt.xlabel("Actual price (THB)")
plt.ylabel("Predicted price (THB)")
plt.title("Random Forest — Actual vs Predicted")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "random_forest_actual_vs_predicted.png", dpi=180)
plt.show()

## Save Model

In [ ]:
model_path = MODEL_DIR / "random_forest_pipeline.joblib"
joblib.dump(best_pipeline, model_path)

metadata = {
    "model": "RandomForestRegressor",
    "target": TARGET,
    "currency": "THB",
    "dataset": "One2Car Thailand",
    "features": FEATURES,
    "validation_metrics": val_metrics,
    "test_metrics": test_metrics,
    "tuning_enabled": RUN_TUNING,
    "n_iter": N_ITER if RUN_TUNING else 0,
    "cv": CV if RUN_TUNING else None,
}

with open(MODEL_DIR / "random_forest_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

loaded = joblib.load(model_path)
reload_pred = loaded.predict(X_test)
max_diff = float(np.max(np.abs(test_pred - reload_pred)))
print("Model:", model_path)
print("Reload max prediction difference:", max_diff)
assert max_diff < 1e-9